# NB02 — Chunking for Regulatory RAG

**Project:** Scientific AI Builder — Build02 Regulation Library  
**Input:** `parsed_regulation_library.json` from NB01  
**Goal:** Compare two chunking strategies before embedding.

이번 Notebook의 핵심 질문은:

> **EU 법령을 단순히 일정 길이로 자르는 것보다 Article / Annex / table 구조를 보존하는 것이 retrieval에 더 유리한가?**

우리는 두 가지 corpus를 만듭니다.

### A. Naive chunking
- 같은 clean source content를 일정 글자 수로 자름
- 법적 구조를 특별히 고려하지 않음

### B. Legal-aware chunking
- Article 경계를 보존
- 긴 Article만 내부에서 분할
- Annex table은 **row + table/header context**로 보존
- amendment history / navigation table은 제외

> 아직 embedding이나 LLM은 사용하지 않습니다.  
> NB02의 목적은 **RAG에 넣을 chunk 자체의 품질을 검증**하는 것입니다.


## 0. Setup

Notebook과 `data/parsed_regulation_library.json`이 다음처럼 있다고 가정합니다.

```text
project/
├─ NB01_regulation_library.ipynb
├─ NB02_chunking.ipynb
└─ data/
   ├─ parsed_regulation_library.json
   └─ ... five EUR-Lex HTML files
```


In [1]:
from pathlib import Path
import json
import re
import pandas as pd
from statistics import mean, median

pd.set_option("display.max_colwidth", 180)

BASE_DIR = Path("data")
INPUT_JSON = __import__("pathlib").Path("data/processed/parsed_regulation_library.json")

print("Input exists:", INPUT_JSON.exists(), INPUT_JSON)


Input exists: True data\parsed_regulation_library.json


## 1. Load parsed Regulation Library


In [2]:
with INPUT_JSON.open("r", encoding="utf-8") as f:
    docs = json.load(f)

print("Documents:", len(docs))

for d in docs:
    print(
        f"- {d['file']} | "
        f"articles={len(d['articles'])} | "
        f"table_rows={len(d['table_rows'])}"
    )


Documents: 5
- EU_2023_915_consolidated_2026-08-19.html | articles=11 | table_rows=590
- EU_2002_178_consolidated_2026-01-01.html | articles=65 | table_rows=13
- EU_2017_625_consolidated_2025-01-05.html | articles=167 | table_rows=751
- EU_2019__1013_original.html | articles=2 | table_rows=7
- EU_2019_1793_consolidated_2026-06-30.html | articles=15 | table_rows=501


## 2. Why we should not embed every HTML table

NB01의 `table_rows`에는 실제 Annex 표뿐 아니라 다음도 포함되어 있습니다.

- amendment history
- Official Journal metadata
- corrigenda
- layout / recital tables
- correlation tables

이런 row를 모두 embedding하면 regulatory retrieval에 **noise**가 들어갈 수 있습니다.

이번 benchmark에서 실제로 중요한 structured Annex는 주로:

- **Regulation (EU) 2023/915** — contaminant maximum-level tables
- **Implementing Regulation (EU) 2019/1793** — country/product/hazard/check-frequency tables

입니다.

따라서 NB02에서는 **이 두 법령의 substantive Annex tables만 table chunks로 사용**합니다.

> 이것은 universal EUR-Lex parser 규칙이 아니라, 현재 Build02 v1 corpus에 맞춘 explicit filtering rule입니다.


In [3]:
def is_substantive_table_row(doc, row):
    """
    Keep only substantive Annex/table rows needed for this Build02 v1 corpus.

    NB01 showed:
    - 2023/915: tables 1-2 = title/amendment history, table 3+ = Annex contaminant tables
    - 2019/1793: tables 1-3 = title/amendment/corrigenda, table 4+ = substantive Annexes
    - other three benchmark laws: current Q1-Q10 needs are covered by Articles, so tables are omitted
    """
    file_name = doc["file"]

    if "2023_915" in file_name:
        return row["table_index"] >= 3

    if "2019_1793" in file_name:
        return row["table_index"] >= 4

    return False


filtered_table_counts = []

for d in docs:
    kept = [r for r in d["table_rows"] if is_substantive_table_row(d, r)]
    filtered_table_counts.append({
        "file": d["file"],
        "all_table_rows": len(d["table_rows"]),
        "kept_table_rows": len(kept),
    })

pd.DataFrame(filtered_table_counts)


,file,all_table_rows,kept_table_rows
0,EU_2023_915_consolidated_2026-08-19.html,590,577
1,EU_2002_178_consolidated_2026-01-01.html,13,0
2,EU_2017_625_consolidated_2025-01-05.html,751,0
3,EU_2019__1013_original.html,7,0
4,EU_2019_1793_consolidated_2026-06-30.html,501,477


## 3. Common text splitter

두 strategy를 공정하게 비교하기 위해 동일한 기본 chunk size를 사용합니다.

기본값:
- `MAX_CHARS = 1800`
- `OVERLAP = 200`

**중요:** Legal-aware 방식에서도 너무 긴 Article은 분할해야 합니다.  
다만 Article 경계를 넘지 않고, 각 subchunk에 동일한 legal metadata를 붙입니다.


In [4]:
MAX_CHARS = 1800
OVERLAP = 200


def clean_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def split_long_text(text, max_chars=MAX_CHARS, overlap=OVERLAP):
    """
    Character-based splitter with word-boundary preference.
    Used only when one source unit is too long.
    """
    text = clean_text(text)

    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + max_chars, len(text))

        # Prefer ending at a whitespace boundary.
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start + int(max_chars * 0.6):
                end = boundary

        part = text[start:end].strip()
        if part:
            chunks.append(part)

        if end >= len(text):
            break

        start = max(end - overlap, start + 1)

    return chunks


## 4. Create a clean common source corpus

Naive와 Legal-aware 비교에서 중요한 것은 **같은 정보량을 사용**하는 것입니다.

따라서 두 방식 모두 다음을 사용합니다.

1. 모든 Articles
2. 위에서 필터링한 substantive Annex/table rows

차이는 **어떻게 묶어서 chunk로 만드는가**입니다.


In [5]:
def source_units_for_doc(doc):
    units = []

    # Articles
    for a in doc["articles"]:
        units.append({
            "unit_type": "article",
            "location": f'{a["article"]} — {a["heading"]}'.strip(" —"),
            "text": clean_text(a["text"]),
        })

    # Substantive table rows
    for row in doc["table_rows"]:
        if is_substantive_table_row(doc, row):
            units.append({
                "unit_type": "table_row",
                "location": f'Table {row["table_index"]}, row {row["row_index"]}',
                "text": clean_text(row["row_text"]),
            })

    return units


common_source_summary = []

for d in docs:
    units = source_units_for_doc(d)
    common_source_summary.append({
        "file": d["file"],
        "articles": sum(u["unit_type"] == "article" for u in units),
        "table_rows": sum(u["unit_type"] == "table_row" for u in units),
        "source_units": len(units),
        "source_chars": sum(len(u["text"]) for u in units),
    })

pd.DataFrame(common_source_summary)


,file,articles,table_rows,source_units,source_chars
0,EU_2023_915_consolidated_2026-08-19.html,11,577,588,106305
1,EU_2002_178_consolidated_2026-01-01.html,65,0,65,93404
2,EU_2017_625_consolidated_2025-01-05.html,167,0,167,299536
3,EU_2019__1013_original.html,2,0,2,1267
4,EU_2019_1793_consolidated_2026-06-30.html,15,477,492,41315


# Part A — Naive fixed-size chunking

Naive 방식은 Article / Annex row metadata를 chunk boundary에 사용하지 않습니다.

문서별로 clean source units를 하나의 긴 text로 이어 붙인 뒤 고정 길이로 자릅니다.

즉, 한 chunk 안에:
- Article 끝 + 다음 Article 시작
- table row 끝 + 다음 row 시작

이 섞일 수 있습니다.


In [6]:
def build_naive_chunks(docs, max_chars=MAX_CHARS, overlap=OVERLAP):
    chunks = []

    for doc in docs:
        units = source_units_for_doc(doc)

        # Same cleaned source content, flattened into one document string.
        doc_text = "\n".join(u["text"] for u in units)
        parts = split_long_text(doc_text, max_chars=max_chars, overlap=overlap)

        for i, part in enumerate(parts, start=1):
            chunks.append({
                "chunk_id": f'naive::{doc["file"]}::{i:04d}',
                "strategy": "naive",
                "file": doc["file"],
                "regulation_title": doc["regulation_title"],
                "document_type": doc["document_type"],
                "version_date": doc["version_date"],
                "canonical_url": doc["canonical_url"],
                "location": f"fixed_chunk_{i}",
                "source_type": "mixed",
                "text": part,
                "char_count": len(part),
            })

    return chunks


naive_chunks = build_naive_chunks(docs)

print("Naive chunks:", len(naive_chunks))
pd.DataFrame(naive_chunks)[
    ["chunk_id", "file", "location", "char_count", "text"]
].head()


Naive chunks: 342


,chunk_id,file,location,char_count,text
0,naive::EU_2023_915_consolidated_2026-08-19.html::0001,EU_2023_915_consolidated_2026-08-19.html,fixed_chunk_1,1794,"Article 1 Definitions For the purposes of this Regulation, the following definitions apply: (a) ‘food’ means food as defined in Article 2 of Regulation (EC) No 178/2002 of the ..."
1,naive::EU_2023_915_consolidated_2026-08-19.html::0002,EU_2023_915_consolidated_2026-08-19.html,fixed_chunk_2,1797,"are integrated so that all incoming lots are cleaned, sorted and processed in the same establishment, the maximum levels shall apply to unprocessed cereals in the production ch..."
2,naive::EU_2023_915_consolidated_2026-08-19.html::0003,EU_2023_915_consolidated_2026-08-19.html,fixed_chunk_3,1791,"e of maximum protection of human health. 3. Where no specific Union maximum levels for food for infants and young children are set out in Annex I, Member States may provide for..."
3,naive::EU_2023_915_consolidated_2026-08-19.html::0004,EU_2023_915_consolidated_2026-08-19.html,fixed_chunk_4,1793,ication code shall be indelibly marked on each individual package of the consignment and on the original accompanying document. 3. Food to be subjected to sorting or other phys...
4,naive::EU_2023_915_consolidated_2026-08-19.html::0005,EU_2023_915_consolidated_2026-08-19.html,fixed_chunk_5,1796,". The exception of groundnuts (peanuts) and other oilseeds for crushing from the application of the maximum levels set out in Annex I, shall only apply to consignments, which: ..."


# Part B — Legal-structure-aware chunking

Legal-aware 방식은 다음 원칙을 사용합니다.

### Articles
- Article 하나를 기본 semantic unit으로 사용
- Article이 너무 길 경우에만 내부 분할
- subchunk마다 Article 번호와 heading metadata 유지

### Annex tables
- row를 독립 unit으로 사용
- 하지만 row만 저장하면 column 의미를 잃을 수 있으므로
  **해당 table의 앞부분(header/context)을 함께 붙임**

예:

```text
Regulation (EU) 2023/915
Table context: 1 Mycotoxins | 1.1 Aflatoxins | Maximum level (μg/kg)

1.1.3 | Dried figs | 6.0 | 10.0 | ...
```


In [7]:
def group_rows_by_table(doc):
    grouped = {}
    for row in doc["table_rows"]:
        if is_substantive_table_row(doc, row):
            grouped.setdefault(row["table_index"], []).append(row)
    return grouped


def build_table_context(rows, max_header_rows=3):
    """
    Preserve genuine table headers/section labels without attaching unrelated
    first data rows (e.g. Bangladesh/Bolivia) as context.
    """
    if not rows:
        return ""

    first_cells = [clean_text(x).lower() for x in rows[0].get("cells", [])]

    # Import-control tables have a single explicit column-header row.
    if any("country of origin" in x for x in first_cells) or (
        any("hazard" in x for x in first_cells)
        and any("frequency of identity" in x for x in first_cells)
    ):
        return clean_text(rows[0]["row_text"])

    # Contaminant tables often use multiple rows for section + column headers.
    header_parts = []
    for row in rows[:max_header_rows]:
        txt = clean_text(row["row_text"])
        first = clean_text(row["cells"][0]).lower() if row.get("cells") else ""
        is_numbered_data_row = bool(re.match(r"^\d+(?:\.\d+)*$", first))
        is_column_header = any(
            marker in txt.lower()
            for marker in ["maximum level", "remarks", "aflatoxins", "mycotoxins"]
        )
        # Keep section labels and column headers, but not ordinary numbered data rows.
        if txt and (not is_numbered_data_row or is_column_header) and txt not in header_parts:
            header_parts.append(txt)

    return " || ".join(header_parts)


def build_turkiye_figs_group_chunk(doc, rows, table_context):
    """
    Reconstruct the logical Annex II, Section 1, Row 14 entry for Türkiye dried-fig
    products. In the 2026-06-30 consolidated HTML, this one logical legal entry is
    spread across physical HTML rows 135-151; the hazard/check-frequency cells
    appear near the end of that block rather than beside the first product row.
    """
    physical_rows = [
        r for r in rows
        if 135 <= int(r.get("row_index", -1)) <= 151
    ]

    raw_rows = " || ".join(
        f'HTML row {r["row_index"]}: {clean_text(r["row_text"])}'
        for r in physical_rows
    )

    text = (
        f'{doc["regulation_title"]}. '
        "Legal location: Annex II, Section 1, logical row 14 — Türkiye (TR). "
        "Product scope: dried figs; mixtures of nuts or dried fruits containing figs; "
        "dried fig paste; dried figs prepared or preserved, including mixtures. "
        "Hazards listed for this logical entry: aflatoxins and ochratoxin A. "
        "Frequency of identity and physical checks: 30%. "
        f"Table column context: {table_context}. "
        f"Underlying consolidated-HTML rows grouped into this one logical entry: {raw_rows}. "
        "The row-group reconstruction is specific to the supplied Regulation (EU) 2019/1793 "
        "consolidated version dated 2026-06-30. Cross-check basis: Commission Implementing "
        "Regulation (EU) 2025/1441, Annex I, row 14."
    )

    return {
        "chunk_id": f'legal::{doc["file"]}::annexII_row14_turkiye_figs',
        "strategy": "legal_aware",
        "file": doc["file"],
        "regulation_title": doc["regulation_title"],
        "document_type": doc["document_type"],
        "version_date": doc["version_date"],
        "canonical_url": doc["canonical_url"],
        "source_type": "table_group",
        "location": "Annex II, Section 1, Row 14 — Türkiye dried-fig products",
        "article_id": None,
        "article": None,
        "heading": None,
        "table_index": 5,
        "row_index": 135,
        "text": text,
        "char_count": len(text),
    }


def build_legal_aware_chunks(docs, max_chars=MAX_CHARS, overlap=OVERLAP):
    chunks = []

    for doc in docs:
        # -------- Articles --------
        for article in doc["articles"]:
            article_text = clean_text(article["text"])
            parts = split_long_text(article_text, max_chars=max_chars, overlap=overlap)

            for part_idx, part in enumerate(parts, start=1):
                chunks.append({
                    "chunk_id": (
                        f'legal::{doc["file"]}::'
                        f'{article["article_id"]}::{part_idx:02d}'
                    ),
                    "strategy": "legal_aware",
                    "file": doc["file"],
                    "regulation_title": doc["regulation_title"],
                    "document_type": doc["document_type"],
                    "version_date": doc["version_date"],
                    "canonical_url": doc["canonical_url"],
                    "source_type": "article",
                    "location": f'{article["article"]} — {article["heading"]}'.strip(" —"),
                    "article_id": article["article_id"],
                    "article": article["article"],
                    "heading": article["heading"],
                    "table_index": None,
                    "row_index": None,
                    "text": part,
                    "char_count": len(part),
                })

        # -------- Annex / substantive tables --------
        grouped = group_rows_by_table(doc)

        for table_index, rows in grouped.items():
            table_context = build_table_context(rows)

            # Specific reconstruction for the target logical Annex II entry.
            # The exact physical row span is validated against the supplied snapshot.
            if (
                "2019_1793" in doc["file"]
                and doc["version_date"] == "2026-06-30"
                and table_index == 5
            ):
                candidate_rows = [
                    r for r in rows
                    if 135 <= int(r.get("row_index", -1)) <= 151
                ]
                candidate_text = " ".join(r["row_text"] for r in candidate_rows).lower()
                valid_group = (
                    len(candidate_rows) == 17
                    and "türkiye" in candidate_rows[0]["row_text"].lower()
                    and "dried figs" in candidate_text
                    and "aflatoxins and" in candidate_text
                    and "ochratoxin a" in candidate_text
                    and any("30" in r.get("cells", []) for r in candidate_rows)
                )

                if valid_group:
                    chunks.append(
                        build_turkiye_figs_group_chunk(doc, rows, table_context)
                    )
                    rows = [
                        r for r in rows
                        if not (135 <= int(r.get("row_index", -1)) <= 151)
                    ]
                else:
                    print(
                        "WARNING: Türkiye dried-fig row-group validation failed. "
                        "Review Annex II table rows before using the corpus."
                    )

            for row in rows:
                row_text = clean_text(row["row_text"])
                first_cell = clean_text(row["cells"][0]) if row.get("cells") else ""
                numbered = bool(re.match(r"^\d+(?:\.\d+)*$", first_cell))

                # Exclude genuine header-only rows, but retain meaningful numbered data.
                low_cells = " ".join(row.get("cells", [])).lower()
                is_column_header = (
                    "country of origin" in low_cells
                    or ("maximum level" in low_cells and "remarks" in low_cells)
                )
                is_very_short_header = (
                    row in rows[:3]
                    and len(row_text) < 140
                    and not numbered
                )

                if is_column_header or is_very_short_header:
                    continue

                full_text = (
                    f'{doc["regulation_title"]}. '
                    f'Table context: {table_context}. '
                    f'Row: {row_text}'
                )

                parts = split_long_text(
                    full_text,
                    max_chars=max_chars,
                    overlap=overlap
                )

                for part_idx, part in enumerate(parts, start=1):
                    chunks.append({
                        "chunk_id": (
                            f'legal::{doc["file"]}::'
                            f'table{table_index}_row{row["row_index"]}::{part_idx:02d}'
                        ),
                        "strategy": "legal_aware",
                        "file": doc["file"],
                        "regulation_title": doc["regulation_title"],
                        "document_type": doc["document_type"],
                        "version_date": doc["version_date"],
                        "canonical_url": doc["canonical_url"],
                        "source_type": "table_row",
                        "location": f'Table {table_index}, row {row["row_index"]}',
                        "article_id": None,
                        "article": None,
                        "heading": None,
                        "table_index": table_index,
                        "row_index": row["row_index"],
                        "text": part,
                        "char_count": len(part),
                    })

    return chunks


legal_chunks = build_legal_aware_chunks(docs)

print("Legal-aware chunks:", len(legal_chunks))
pd.DataFrame(legal_chunks)[
    ["chunk_id", "file", "source_type", "location", "char_count", "text"]
].head()


Legal-aware chunks: 1383


,chunk_id,file,source_type,location,char_count,text
0,legal::EU_2023_915_consolidated_2026-08-19.html::art_1::01,EU_2023_915_consolidated_2026-08-19.html,article,Article 1 — Definitions,982,"Article 1 Definitions For the purposes of this Regulation, the following definitions apply: (a) ‘food’ means food as defined in Article 2 of Regulation (EC) No 178/2002 of the ..."
1,legal::EU_2023_915_consolidated_2026-08-19.html::art_2::01,EU_2023_915_consolidated_2026-08-19.html,article,Article 2 — General rules,835,Article 2 General rules 1. The food listed in Annex I shall not be placed on the market and shall not be used as a raw material in food or as an ingredient in food where it con...
2,legal::EU_2023_915_consolidated_2026-08-19.html::art_3::01,EU_2023_915_consolidated_2026-08-19.html,article,"Article 3 — Dried, diluted, processed and compound food",1588,"Article 3 Dried, diluted, processed and compound food 1. Where no specific Union maximum levels are set out in Annex I for food which is dried, diluted, processed or compound f..."
3,legal::EU_2023_915_consolidated_2026-08-19.html::art_4::01,EU_2023_915_consolidated_2026-08-19.html,article,Article 4 — Prohibition on detoxification,147,Article 4 Prohibition on detoxification Food containing contaminants listed in Annex I shall not be deliberately detoxified by chemical treatments.
4,legal::EU_2023_915_consolidated_2026-08-19.html::art_5::01,EU_2023_915_consolidated_2026-08-19.html,article,Article 5 — Food to be subjected to sorting or other physical treatment before placing on the market for the final consumer or use as a food ingredient,1793,Article 5 Food to be subjected to sorting or other physical treatment before placing on the market for the final consumer or use as a food ingredient 1. Where a maximum level f...


## 5. Compare chunk statistics

여기서는 어느 방식이 더 '좋다'고 결론내리지 않습니다.

우선 확인할 것은:
- chunk 개수
- 평균/중앙 길이
- 지나치게 짧은 chunk 수
- 최대 길이

실제 retrieval 성능 비교는 다음 Notebook에서 embedding 후 진행합니다.


In [8]:
def chunk_stats(name, chunks):
    lengths = [c["char_count"] for c in chunks]

    return {
        "strategy": name,
        "n_chunks": len(chunks),
        "mean_chars": round(mean(lengths), 1),
        "median_chars": round(median(lengths), 1),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
        "chunks_under_200_chars": sum(x < 200 for x in lengths),
    }


stats_df = pd.DataFrame([
    chunk_stats("naive", naive_chunks),
    chunk_stats("legal_aware", legal_chunks),
])

stats_df


,strategy,n_chunks,mean_chars,median_chars,min_chars,max_chars,chunks_under_200_chars
0,naive,342,1785.0,1796.0,308,1799,0
1,legal_aware,1383,640.4,380.0,115,1799,61


## 6. Simple lexical search for inspection

아직 vector search가 아니므로 성능평가용이 아닙니다.

목적은 단순합니다.

> **핵심 benchmark 정보가 어떤 chunk 모습으로 저장되었는지 눈으로 확인**

합니다.


In [9]:
def lexical_search(query, chunks, top_k=10):
    terms = re.findall(r"[A-Za-z0-9À-ž]+", query.lower())
    results = []

    for c in chunks:
        text_low = c["text"].lower()
        score = sum(text_low.count(term) for term in terms if len(term) > 1)

        if score > 0:
            results.append({
                "score": score,
                "chunk_id": c["chunk_id"],
                "file": c["file"],
                "source_type": c["source_type"],
                "location": c["location"],
                "text": c["text"],
            })

    results.sort(key=lambda x: x["score"], reverse=True)
    return pd.DataFrame(results[:top_k])


def compare_search(query, top_k=5):
    print("=" * 100)
    print("QUERY:", query)

    print("\n--- NAIVE ---")
    display(lexical_search(query, naive_chunks, top_k=top_k))

    print("\n--- LEGAL-AWARE ---")
    display(lexical_search(query, legal_chunks, top_k=top_k))


## 7. Diagnostic A — Dried figs / aflatoxin limits

우리가 원하는 legal-aware chunk는 ideally 다음 정보가 한 덩어리로 있어야 합니다.

- Regulation 2023/915
- Aflatoxins
- Dried figs
- B1 = 6.0
- Total = 10.0
- 80% remark / Article 3 reference


In [10]:
compare_search("dried figs aflatoxin 6 10", top_k=8)


QUERY: dried figs aflatoxin 6 10

--- NAIVE ---


,score,chunk_id,file,source_type,location,text
0,37,naive::EU_2019_1793_consolidated_2026-06-30.html::0021,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_21,ex 2008 19 19 | 49 ex 2008 19 99 | 49 Sesamum seeds ( Food and feed ) | 1207 40 90 | Pesticide residues ( 9 ) | 20 ex 2008 19 19 | 49 ex 2008 19 99 | 49 Nutmeg (Food – dried sp...
1,27,naive::EU_2019_1793_consolidated_2026-06-30.html::0017,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_17,"3 ) | 50 ex 0710 22 00 | 10 Peppers of the genus Capsicum (sweet or other than sweet) (Food – dried, rosted, crushed or ground) | 0904 21 10 | Aflatoxins | 50 ex 0904 21 90 | 2..."
2,23,naive::EU_2019_1793_consolidated_2026-06-30.html::0023,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_23,| Salmonella ( 5 ) | 50 (Food) | ex 2008 19 19 | 49 ex 2008 19 99 | 49 14 | Türkiye (TR) | — Dried figs | 0804 20 90 — Mixtures of nuts or dried fruits containing figs | ex 081...
3,21,naive::EU_2019_1793_consolidated_2026-06-30.html::0014,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_14,"inican Republic (DO) | — Sweet peppers ( Capsicum annuum ) | 0709 60 10 | Pesticide residues ( 3 ) ( 16 ) | 50 — Peppers of the genus Capsicum (other than sweet) (Food – fresh,..."
4,20,naive::EU_2019_1793_consolidated_2026-06-30.html::0016,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_16,lled or frozen) | ex 0709 99 90 | 20 | Pesticide residues ( 3 ) ( 7 ) | 30 ex 0710 80 95 | 30 Rice (Food) | 1006 | Pesticide residues ( 3 ) | 10 Guava ( Psidium guajava ) (Food...
5,19,naive::EU_2019_1793_consolidated_2026-06-30.html::0022,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_22,"containing botanicals ( 12 ) (Food) | ex 1302 ex 2106 | Pesticide residues ( 9 ) | 10 Drumsticks ( Moringa oleifera ) (Food fresh, chilled or frozen) | ex 0709 99 90 ex 0710 80..."
6,18,naive::EU_2023_915_consolidated_2026-08-19.html::0012,EU_2023_915_consolidated_2026-08-19.html,mixed,fixed_chunk_12,into force This Regulation shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union . 1 | Mycotoxins 1.1 | Af...
7,18,naive::EU_2023_915_consolidated_2026-08-19.html::0032,EU_2023_915_consolidated_2026-08-19.html,mixed,fixed_chunk_32,"alkaloid containing plants. 2.4.3 | Borage, lovage, marjoram and oregano (dried product) and mixtures exclusively composed of these dried herbs | 1 000 | Without prejudice to m..."



--- LEGAL-AWARE ---


,score,chunk_id,file,source_type,location,text
0,19,legal::EU_2019_1793_consolidated_2026-06-30.html::annexII_row14_turkiye_figs,EU_2019_1793_consolidated_2026-06-30.html,table_group,"Annex II, Section 1, Row 14 — Türkiye dried-fig products","COMMISSION IMPLEMENTING REGULATION (EU) 2019/1793. Legal location: Annex II, Section 1, logical row 14 — Türkiye (TR). Product scope: dried figs; mixtures of nuts or dried frui..."
1,11,legal::EU_2023_915_consolidated_2026-08-19.html::table3_row6::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 3, row 6","COMMISSION REGULATION (EU) 2023/915. Table context: 1 | Mycotoxins || 1.1 | Aflatoxins | Maximum level (μg/kg) | Remarks || B 1 | Sum of B 1 , B 2 , G 1 and G 2 | M 1 | For the..."
2,7,legal::EU_2023_915_consolidated_2026-08-19.html::table3_row5::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 3, row 5","COMMISSION REGULATION (EU) 2023/915. Table context: 1 | Mycotoxins || 1.1 | Aflatoxins | Maximum level (μg/kg) | Remarks || B 1 | Sum of B 1 , B 2 , G 1 and G 2 | M 1 | For the..."
3,7,legal::EU_2023_915_consolidated_2026-08-19.html::table3_row17::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 3, row 17","COMMISSION REGULATION (EU) 2023/915. Table context: 1 | Mycotoxins || 1.1 | Aflatoxins | Maximum level (μg/kg) | Remarks || B 1 | Sum of B 1 , B 2 , G 1 and G 2 | M 1 | For the..."
4,7,legal::EU_2017_625_consolidated_2025-01-05.html::art_103::01,EU_2017_625_consolidated_2025-01-05.html,article,Article 103 — Liaison bodies,Article 103 Liaison bodies 1. Each Member State shall designate one or more liaison bodies acting as contact points responsible for facilitating the exchange of communications ...
5,6,legal::EU_2023_915_consolidated_2026-08-19.html::table15_row6::02,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 15, row 6","analysis, shall be quantified and included in the sum. | For the pyrrolizidine alkaloids, maximum levels refer to lower bound concentrations, which are calculated on the assump..."
6,6,legal::EU_2019_1793_consolidated_2026-06-30.html::table5_row88::01,EU_2019_1793_consolidated_2026-06-30.html,table_row,"Table 5, row 88",COMMISSION IMPLEMENTING REGULATION (EU) 2019/1793. Table context: Row | Country of origin | Food and feed (intended use) | CN code ( 1 ) | TARIC sub-division | Hazard | Frequen...
7,5,legal::EU_2023_915_consolidated_2026-08-19.html::table3_row9::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 3, row 9","COMMISSION REGULATION (EU) 2023/915. Table context: 1 | Mycotoxins || 1.1 | Aflatoxins | Maximum level (μg/kg) | Remarks || B 1 | Sum of B 1 , B 2 , G 1 and G 2 | M 1 | For the..."


## 8. Directly inspect the `1.1.3 Dried figs` legal-aware chunk


In [11]:
fig_chunks = [
    c for c in legal_chunks
    if "dried figs" in c["text"].lower()
    and "1.1.3" in c["text"]
]

for c in fig_chunks[:5]:
    print("=" * 100)
    print(c["chunk_id"])
    print(c["location"])
    print(c["text"])


legal::EU_2023_915_consolidated_2026-08-19.html::table3_row6::01
Table 3, row 6
COMMISSION REGULATION (EU) 2023/915. Table context: 1 | Mycotoxins || 1.1 | Aflatoxins | Maximum level (μg/kg) | Remarks || B 1 | Sum of B 1 , B 2 , G 1 and G 2 | M 1 | For the sum of aflatoxins, maximum levels refer to lower bound concentrations, which are calculated on the assumption that all the values below the limit of quantification are zero.. Row: 1.1.3 | Dried figs | 6,0 | 10,0 | - | In the case of food consisting of dried figs used as only ingredient or in the case of processed products consisting at least of 80 % from dried figs, the maximum levels as established for dried figs apply also to those products. In other cases, Article 3(1) and (2) apply.


## 9. Diagnostic B — Türkiye dried figs import controls

확인하고 싶은 정보:
- Türkiye
- dried figs
- hazard
- aflatoxins / ochratoxin A
- frequency of identity and physical checks


In [12]:
compare_search("Türkiye dried figs aflatoxins ochratoxin", top_k=10)


QUERY: Türkiye dried figs aflatoxins ochratoxin

--- NAIVE ---


,score,chunk_id,file,source_type,location,text
0,18,naive::EU_2023_915_consolidated_2026-08-19.html::0032,EU_2023_915_consolidated_2026-08-19.html,mixed,fixed_chunk_32,"alkaloid containing plants. 2.4.3 | Borage, lovage, marjoram and oregano (dried product) and mixtures exclusively composed of these dried herbs | 1 000 | Without prejudice to m..."
1,16,naive::EU_2023_915_consolidated_2026-08-19.html::0012,EU_2023_915_consolidated_2026-08-19.html,mixed,fixed_chunk_12,into force This Regulation shall enter into force on the twentieth day following that of its publication in the Official Journal of the European Union . 1 | Mycotoxins 1.1 | Af...
2,14,naive::EU_2019_1793_consolidated_2026-06-30.html::0021,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_21,ex 2008 19 19 | 49 ex 2008 19 99 | 49 Sesamum seeds ( Food and feed ) | 1207 40 90 | Pesticide residues ( 9 ) | 20 ex 2008 19 19 | 49 ex 2008 19 99 | 49 Nutmeg (Food – dried sp...
3,14,naive::EU_2019_1793_consolidated_2026-06-30.html::0023,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_23,| Salmonella ( 5 ) | 50 (Food) | ex 2008 19 19 | 49 ex 2008 19 99 | 49 14 | Türkiye (TR) | — Dried figs | 0804 20 90 — Mixtures of nuts or dried fruits containing figs | ex 081...
4,12,naive::EU_2023_915_consolidated_2026-08-19.html::0017,EU_2023_915_consolidated_2026-08-19.html,mixed,fixed_chunk_17,"in shell’ are analysed, it is assumed, when calculating the ochratoxin A content, that all the contamination is on the edible part. 1.2.4 | Pistachios placed on the market for ..."
5,10,naive::EU_2023_915_consolidated_2026-08-19.html::0016,EU_2023_915_consolidated_2026-08-19.html,mixed,fixed_chunk_16,"nd black pepper) Nutmeg ( Myristica fragrans ) Turmeric ( Curcuma longa ) Mixtures of dried spices containing one or more of the abovementioned dried spices | 5,0 | 10,0 | - 1...."
6,8,naive::EU_2023_915_consolidated_2026-08-19.html::0028,EU_2023_915_consolidated_2026-08-19.html,mixed,fixed_chunk_28,"er bound concentrations, which are calculated on the assumption that all the values below the limit of quantification are zero. 2.2.2 | Unprocessed millet grains and sorghum gr..."
7,8,naive::EU_2019_1793_consolidated_2026-06-30.html::0022,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_22,"containing botanicals ( 12 ) (Food) | ex 1302 ex 2106 | Pesticide residues ( 9 ) | 10 Drumsticks ( Moringa oleifera ) (Food fresh, chilled or frozen) | ex 0709 99 90 ex 0710 80..."
8,7,naive::EU_2019_1793_consolidated_2026-06-30.html::0017,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_17,"3 ) | 50 ex 0710 22 00 | 10 Peppers of the genus Capsicum (sweet or other than sweet) (Food – dried, rosted, crushed or ground) | 0904 21 10 | Aflatoxins | 50 ex 0904 21 90 | 2..."
9,6,naive::EU_2019_1793_consolidated_2026-06-30.html::0020,EU_2019_1793_consolidated_2026-06-30.html,mixed,fixed_chunk_20,"40 ex 2008 19 19 | 50 ex 2008 19 92 | 40 | Aflatoxins | 30 ex 2008 19 95 | 40 ex 2008 19 99 | 50 — Oilcake and other solid residues, whether or not ground or in the form of pel..."



--- LEGAL-AWARE ---


,score,chunk_id,file,source_type,location,text
0,20,legal::EU_2019_1793_consolidated_2026-06-30.html::annexII_row14_turkiye_figs,EU_2019_1793_consolidated_2026-06-30.html,table_group,"Annex II, Section 1, Row 14 — Türkiye dried-fig products","COMMISSION IMPLEMENTING REGULATION (EU) 2019/1793. Legal location: Annex II, Section 1, logical row 14 — Türkiye (TR). Product scope: dried figs; mixtures of nuts or dried frui..."
1,10,legal::EU_2023_915_consolidated_2026-08-19.html::table3_row6::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 3, row 6","COMMISSION REGULATION (EU) 2023/915. Table context: 1 | Mycotoxins || 1.1 | Aflatoxins | Maximum level (μg/kg) | Remarks || B 1 | Sum of B 1 , B 2 , G 1 and G 2 | M 1 | For the..."
2,7,legal::EU_2023_915_consolidated_2026-08-19.html::table3_row5::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 3, row 5","COMMISSION REGULATION (EU) 2023/915. Table context: 1 | Mycotoxins || 1.1 | Aflatoxins | Maximum level (μg/kg) | Remarks || B 1 | Sum of B 1 , B 2 , G 1 and G 2 | M 1 | For the..."
3,6,legal::EU_2023_915_consolidated_2026-08-19.html::table3_row17::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 3, row 17","COMMISSION REGULATION (EU) 2023/915. Table context: 1 | Mycotoxins || 1.1 | Aflatoxins | Maximum level (μg/kg) | Remarks || B 1 | Sum of B 1 , B 2 , G 1 and G 2 | M 1 | For the..."
4,6,legal::EU_2023_915_consolidated_2026-08-19.html::table15_row6::02,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 15, row 6","analysis, shall be quantified and included in the sum. | For the pyrrolizidine alkaloids, maximum levels refer to lower bound concentrations, which are calculated on the assump..."
5,5,legal::EU_2023_915_consolidated_2026-08-19.html::table15_row10::02,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 15, row 10","analysis, shall be quantified and included in the sum. | For the pyrrolizidine alkaloids, maximum levels refer to lower bound concentrations, which are calculated on the assump..."
6,5,legal::EU_2023_915_consolidated_2026-08-19.html::table32_row8::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 32, row 8",COMMISSION REGULATION (EU) 2023/915. Table context: 6.3 | Perchlorate | Maximum level (mg/kg) | Remarks. Row: 6.3.2 | Tea ( Camellia sinensis ) (dried product) Herbal and fruit...
7,4,legal::EU_2023_915_consolidated_2026-08-19.html::table4_row3::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 4, row 3","COMMISSION REGULATION (EU) 2023/915. Table context: 1.2 | Ochratoxin A | Maximum level (μg/kg) | Remarks. Row: 1.2.1.1 | Dried vine fruits (currants, raisins and sultanas) and ..."
8,4,legal::EU_2023_915_consolidated_2026-08-19.html::table4_row10::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 4, row 10","COMMISSION REGULATION (EU) 2023/915. Table context: 1.2 | Ochratoxin A | Maximum level (μg/kg) | Remarks. Row: 1.2.7 | Marshmallow roots (dried), dandelion roots (dried) and or..."
9,4,legal::EU_2023_915_consolidated_2026-08-19.html::table4_row18::01,EU_2023_915_consolidated_2026-08-19.html,table_row,"Table 4, row 18",COMMISSION REGULATION (EU) 2023/915. Table context: 1.2 | Ochratoxin A | Maximum level (μg/kg) | Remarks. Row: 1.2.11.2 | products containing at least 20 % dried vine fruits an...


## Diagnostic B2 — Validate the logical Türkiye dried-fig entry

The consolidated HTML spreads the logical Annex II row 14 across multiple physical table rows.
This check verifies that the legal-aware corpus has one grouped chunk with the product scope,
hazards, and identity/physical-check frequency together.

In [13]:
turkiye_fig_group = [
    c for c in legal_chunks
    if c.get("source_type") == "table_group"
    and "2019_1793" in c["file"]
    and "annexII_row14_turkiye_figs" in c["chunk_id"]
]

print("Türkiye dried-fig logical entry chunks:", len(turkiye_fig_group))

if len(turkiye_fig_group) != 1:
    raise AssertionError("Expected exactly one grouped Annex II row 14 chunk.")

target = turkiye_fig_group[0]
required_phrases = [
    "Türkiye (TR)",
    "dried figs",
    "Aflatoxins and ochratoxin A",
    "30%",
    "0804",
]
missing = [phrase for phrase in required_phrases if phrase.lower() not in target["text"].lower()]

if missing:
    raise AssertionError(f"Grouped entry is missing expected content: {missing}")

print("PASS: product scope, hazards, and 30% identity/physical-check frequency are grouped.")
print(target["location"])
print(target["text"])


Türkiye dried-fig logical entry chunks: 1
PASS: product scope, hazards, and 30% identity/physical-check frequency are grouped.
Annex II, Section 1, Row 14 — Türkiye dried-fig products
COMMISSION IMPLEMENTING REGULATION (EU) 2019/1793. Legal location: Annex II, Section 1, logical row 14 — Türkiye (TR). Product scope: dried figs; mixtures of nuts or dried fruits containing figs; dried fig paste; dried figs prepared or preserved, including mixtures. Hazards listed for this logical entry: aflatoxins and ochratoxin A. Frequency of identity and physical checks: 30%. Table column context: Row | Country of origin | Food and feed (intended use) | CN code ( 1 ) | TARIC sub-division | Hazard | Frequency of identity and physical checks (%). Underlying consolidated-HTML rows grouped into this one logical entry: HTML row 135: 14 | Türkiye (TR) | — Dried figs | 0804 20 90 || HTML row 136: — Mixtures of nuts or dried fruits containing figs | ex 0813 50 99 | 50 || HTML row 137: — Dried fig paste | ex 2

## 10. Diagnostic C — Article 19 / withdrawal-recall

178/2002 Article 19의 책임이 Article 단위 metadata와 함께 보존되는지 확인합니다.


In [14]:
article19_chunks = [
    c for c in legal_chunks
    if "2002_178" in c["file"]
    and c["source_type"] == "article"
    and re.match(r"Article\s+19\b", str(c.get("article", "")).replace("\xa0", " "))
]

for c in article19_chunks:
    print("=" * 100)
    print(c["chunk_id"])
    print(c["location"])
    print(c["text"][:2500])


legal::EU_2002_178_consolidated_2026-01-01.html::art_19::01
Article 19 — Responsibilities for food: food business operators
Article 19 Responsibilities for food: food business operators 1. If a food business operator considers or has reason to believe that a food which it has imported, produced, processed, manufactured or distributed is not in compliance with the food safety requirements, it shall immediately initiate procedures to withdraw the food in question from the market where the food has left the immediate control of that initial food business operator and inform the competent authorities thereof. Where the product may have reached the consumer, the operator shall effectively and accurately inform the consumers of the reason for its withdrawal, and if necessary, recall from consumers products already supplied to them when other measures are not sufficient to achieve a high level of health protection. 2. A food business operator responsible for retail or distribution activities 

## 11. Diagnostic D — Prior notification

2019/1013 Article 1에서 다음이 보존되어야 합니다.

- at least one working day before arrival
- logistical constraint derogation
- at least four hours


In [15]:
prior_chunks = [
    c for c in legal_chunks
    if "2019__1013" in c["file"]
    and c["source_type"] == "article"
]

for c in prior_chunks:
    print("=" * 100)
    print(c["chunk_id"])
    print(c["location"])
    print(c["text"])


legal::EU_2019__1013_original.html::art_1::01
Article 1 — Prior notification of consignments
Article 1 Prior notification of consignments 1. The operator responsible for a consignment falling within the categories of animals and goods referred to in Article 47(1) of Regulation (EU) 2017/625 shall give prior notification, to the competent authority of the border control post of first arrival into the Union, at least one working day before the expected arrival of the consignment. 2. By way of derogation from paragraph 1, where logistical constraints prevent compliance with the time limit set out in that paragraph, the competent authorities of the border control posts may apply a period of prior notification of at least four hours before the expected arrival of the consignment. 3. By way of derogation from paragraph 1, the competent authorities of border control posts designated for imports of unprocessed logs and sawn and chipped wood in accordance with Article 4 of Commission Delegated 

## 12. Check for obvious noise

우리가 제외하려고 한 amendment-history 문자열이 legal-aware table chunks에 과도하게 들어갔는지 간단히 확인합니다.

`►M1`, `Official Journal`, `Corrigendum` 등이 일부 Article text 내부 amendment marker로 남아 있을 수는 있습니다.  
하지만 **table chunk 자체가 amendment-history row로 생성되는 것은 피하는 것**이 목적입니다.


In [16]:
noise_patterns = ["official journal", "corrigendum", "►m1", "►m2"]

noise_rows = []

for c in legal_chunks:
    if c["source_type"] != "table_row":
        continue

    low = c["text"].lower()

    hits = [p for p in noise_patterns if p in low]

    if hits:
        noise_rows.append({
            "chunk_id": c["chunk_id"],
            "file": c["file"],
            "location": c["location"],
            "hits": hits,
            "preview": c["text"][:300],
        })

noise_df = pd.DataFrame(noise_rows)

print("Potential noisy TABLE chunks:", len(noise_df))
noise_df.head(20)


Potential noisy TABLE chunks: 0


""


## 13. Build a compact chunk manifest

Embedding 단계에서 각 chunk에는 text뿐 아니라 provenance metadata가 반드시 있어야 합니다.

최소 metadata:
- regulation title
- source file
- document type
- version date
- canonical URL
- source type
- legal location


In [17]:
manifest_df = pd.DataFrame([
    {
        "chunk_id": c["chunk_id"],
        "strategy": c["strategy"],
        "file": c["file"],
        "regulation_title": c["regulation_title"],
        "version_date": c["version_date"],
        "source_type": c["source_type"],
        "location": c["location"],
        "char_count": c["char_count"],
    }
    for c in legal_chunks
])

manifest_df.head(20)


,chunk_id,strategy,file,regulation_title,version_date,source_type,location,char_count
0,legal::EU_2023_915_consolidated_2026-08-19.html::art_1::01,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,Article 1 — Definitions,982
1,legal::EU_2023_915_consolidated_2026-08-19.html::art_2::01,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,Article 2 — General rules,835
2,legal::EU_2023_915_consolidated_2026-08-19.html::art_3::01,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,"Article 3 — Dried, diluted, processed and compound food",1588
3,legal::EU_2023_915_consolidated_2026-08-19.html::art_4::01,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,Article 4 — Prohibition on detoxification,147
4,legal::EU_2023_915_consolidated_2026-08-19.html::art_5::01,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,Article 5 — Food to be subjected to sorting or other physical treatment before placing on the market for the final consumer or use as a food ingredient,1793
5,legal::EU_2023_915_consolidated_2026-08-19.html::art_5::02,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,Article 5 — Food to be subjected to sorting or other physical treatment before placing on the market for the final consumer or use as a food ingredient,363
6,legal::EU_2023_915_consolidated_2026-08-19.html::art_6::01,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,"Article 6 — Labelling provisions for groundnuts (peanuts), other oilseeds, derived products thereof and cereals",1345
7,legal::EU_2023_915_consolidated_2026-08-19.html::art_7::01,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,Article 7 — Derogations from Article 2,1793
8,legal::EU_2023_915_consolidated_2026-08-19.html::art_7::02,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,Article 7 — Derogations from Article 2,1797
9,legal::EU_2023_915_consolidated_2026-08-19.html::art_7::03,legal_aware,EU_2023_915_consolidated_2026-08-19.html,COMMISSION REGULATION (EU) 2023/915,2026-08-19,article,Article 7 — Derogations from Article 2,1798


## 14. Save both corpora

다음 Notebook에서 같은 embedding model을 사용해 두 corpus를 각각 index한 뒤 retrieval을 비교할 예정입니다.


In [18]:
NAIVE_OUT = __import__("pathlib").Path("data/processed/chunks_naive.json")
LEGAL_OUT = __import__("pathlib").Path("data/processed/chunks_legal_aware.json")

with NAIVE_OUT.open("w", encoding="utf-8") as f:
    json.dump(naive_chunks, f, ensure_ascii=False, indent=2)

with LEGAL_OUT.open("w", encoding="utf-8") as f:
    json.dump(legal_chunks, f, ensure_ascii=False, indent=2)

print("Saved:", NAIVE_OUT)
print("Saved:", LEGAL_OUT)


Saved: data\chunks_naive.json
Saved: data\chunks_legal_aware.json


# NB02 completion checklist

- [ ] `parsed_regulation_library.json` loaded
- [ ] substantive table filtering reviewed
- [ ] naive chunks created
- [ ] legal-aware chunks created
- [ ] chunk statistics compared
- [ ] `1.1.3 Dried figs` chunk preserves 6.0 / 10.0 + context
- [ ] Türkiye dried figs row is retrievable
- [ ] 178/2002 Article 19 is preserved with Article metadata
- [ ] 2019/1013 Article 1 preserves one-working-day / four-hour rules
- [ ] amendment-history table noise is limited
- [ ] `chunks_naive.json` saved
- [ ] `chunks_legal_aware.json` saved

---

# Next — NB03: Embedding & Retrieval Benchmark

다음 단계에서는 **LLM을 아직 연결하지 않고 retrieval만 테스트**하는 것이 좋습니다.

예정:

1. free Hugging Face embedding model
2. FAISS index
3. 동일한 benchmark questions Q1–Q10 사용
4. naive vs legal-aware retrieval 비교
5. Top-k retrieved chunks 확인
6. retrieval hit / legal grounding 분석

즉 Build02를 다음처럼 나눕니다.

**NB01 Parsing → NB02 Chunking → NB03 Embedding/Retrieval → NB04 LLM RAG Answering → Benchmark**
